# 02 -- Grouped Split, Baseline Benchmarking & Model Training

**Project:** FlyRank Ranking Signal Analysis -- Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 02 of 03 (Model Training & Benchmark Optimization)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
1. **Grouped Holdout Split Design:** Enforce a strict zero-client-leakage partition on `client_id` with programmatic disjointness assertions.
2. **Transparent Heuristic Baselines:** Fit Baseline 1 (Mean Dummy Regressor) and Baseline 2 (Univariate Word Count Rule) to establish the floor to beat.
3. **High-Accuracy Model Training:** 
   - Fit candidate algorithms (cross-validated Polynomial Interaction Ridge, HistGradientBoosting Regressor, and Stacking Ensemble).
   - Maximize generalization accuracy on unseen client domains.
4. **Ablation & Leakage Sanity:** Quantify post-ranking signal influence and demonstrate why grouped splitting is strictly necessary.
5. **Payload Export:** Export model predictions, test set ground truth, and metadata to `work/outputs/model_evaluation_payload.pkl` for evaluation in Notebook 03.

## 1. Grouped Holdout Split Design (Zero Client Leakage)

### 1.1 The Multi-Tenant Generalization Challenge
In multi-tenant SaaS platforms, standard random row splits produce severe **client leakage**: content items from the same client share identical domain authority, root backlink velocity, CMS infrastructure, and brand equity. A random split allows models to memorize client-level baseline volume, inflating test performance while collapsing on brand-new client domains.

### 1.2 The Client Holdout Standard
To guarantee 100% scientific validity:
* We partition strictly by `client_id` using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`.
* We assert that training clients and test clients are mutually exclusive:
  $$\text{set}(\text{clients}_{\text{train}}) \cap \text{set}(\text{clients}_{\text{test}}) = \emptyset$$
* Test metrics reflect true generalization performance on entirely unseen enterprise domains.

In [1]:
# ============================================================
# Code Cell 1: Load Features and Execute Grouped Split (Step 2.1)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle, pathlib, os, warnings, sys
from scipy import stats
warnings.filterwarnings('ignore')
if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

curr = pathlib.Path.cwd().resolve()
while curr.name != 'DA-Project' and curr != curr.parent:
    curr = curr.parent
PROJECT_ROOT = curr
OUTPUT_DIR = PROJECT_ROOT / 'work' / 'outputs'

# Load prepared features
df = pd.read_parquet(OUTPUT_DIR / 'prepared_features.parquet')
print(f"Loaded features: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Separate target, groups, and feature matrix
META_COLS = ['client_id', 'content_id', 'target_log_impressions']
feature_cols = [c for c in df.columns if c not in META_COLS]
X = df[feature_cols].copy()
y = df['target_log_impressions'].copy()
groups = df['client_id'].copy()

# GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Assert disjoint clients
train_clients = set(groups_train.unique())
test_clients = set(groups_test.unique())
assert train_clients.isdisjoint(test_clients), "LEAKAGE: Train and test client sets overlap!"

print(f"\n[PASS] GroupShuffleSplit complete. Client sets are DISJOINT.")
print(f"  Train: {len(train_clients)} clients, {len(X_train):,} rows")
print(f"  Test:  {len(test_clients)} clients, {len(X_test):,} rows")
print(f"  Train clients: {sorted(train_clients)}")
print(f"  Test clients:  {sorted(test_clients)}")
print(f"\n  Train target mean: {y_train.mean():.4f}, std: {y_train.std():.4f}")
print(f"  Test  target mean: {y_test.mean():.4f}, std: {y_test.std():.4f}")

Loaded features: 30,000 rows x 52 columns

[PASS] GroupShuffleSplit complete. Client sets are DISJOINT.
  Train: 25 clients, 23,837 rows
  Test:  7 clients, 6,163 rows
  Train clients: ['client_02d20bbd7e', 'client_0b918943df', 'client_19581e27de', 'client_1a6562590e', 'client_25fc0e7096', 'client_2c624232cd', 'client_349c41201b', 'client_3fdba35f04', 'client_4ec9599fc2', 'client_4fc82b26ae', 'client_6208ef0f77', 'client_624b60c58c', 'client_7f2253d7e2', 'client_8722616204', 'client_9400f1b21c', 'client_98a3ab7c34', 'client_9f14025af0', 'client_a88a7902cb', 'client_b4944c6ff0', 'client_bbb965ab0c', 'client_d029fa3a95', 'client_d4735e3a26', 'client_d59eced1de', 'client_e29c9c180c', 'client_f74efabef1']
  Test clients:  ['client_434c9b5ae5', 'client_4e07408562', 'client_8527a891e2', 'client_8b940be7fb', 'client_bdd2d3af3a', 'client_e629fa6598', 'client_f369cb89fc']

  Train target mean: 6.3089, std: 2.6770
  Test  target mean: 5.7238, std: 2.6828


## 2. Transparent Heuristic Baseline Benchmarks

To verify that machine learning provides genuine predictive lift rather than fitting noise, evaluate two non-ML baselines on the exact same test cohort:
1. **Baseline 1 (Mean Dummy Regressor):** Always predicts the training set target mean $\bar{y}_{\text{train}}$.
2. **Baseline 2 (Univariate Word Count Rule):** A simple single-variable OLS regression on `log_word_count`, reflecting the common SEO belief that "longer articles rank higher and gain more visibility".

In [2]:
# ============================================================
# Code Cell 2: Baseline Benchmarks (Step 2.2)
# ============================================================

def evaluate_model(y_true, y_pred, name):
    """Compute standard regression metrics."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    # Handle constant predictions gracefully for Spearman rho
    if np.all(y_pred == y_pred[0]):
        spearman_r = 0.0
    else:
        spearman_r, _ = stats.spearmanr(y_true, y_pred)
        if np.isnan(spearman_r):
            spearman_r = 0.0
    return {'model': name, 'R2': round(float(r2), 4), 'RMSE': round(float(rmse), 4),
            'MAE': round(float(mae), 4), 'Spearman': round(float(spearman_r), 4)}

# --- Baseline 1: Mean Dummy Regressor ---
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)
metrics_dummy = evaluate_model(y_test, pred_dummy, 'Baseline 1: Mean Dummy')

# --- Baseline 2: OLS on log_word_count ---
ols = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('ols', LinearRegression())
])
ols.fit(X_train[['log_word_count']], y_train)
pred_ols = ols.predict(X_test[['log_word_count']])
metrics_ols = evaluate_model(y_test, pred_ols, 'Baseline 2: OLS log_word_count')

print("=" * 70)
print("BASELINE BENCHMARK RESULTS")
print("=" * 70)
baseline_df = pd.DataFrame([metrics_dummy, metrics_ols])
print(baseline_df.to_string(index=False))


BASELINE BENCHMARK RESULTS
                         model      R2   RMSE    MAE  Spearman
        Baseline 1: Mean Dummy -0.0476 2.7457 2.2757    0.0000
Baseline 2: OLS log_word_count -0.0471 2.7451 2.2859   -0.0571


## 3. High-Accuracy Model Training

### 3.1 Model Architecture Strategy
1. **Regularized Linear Models with Interactions (Candidate A: `RidgeCV`):** Pairwise interaction terms across primary signals regularized with cross-validated L2 penalty.
2. **Gradient Boosted Decision Trees (Candidate B: `HistGradientBoostingRegressor`):** Captures non-linear thresholds and step-function relationships, tuned via `GroupKFold` cross-validation on TRAIN clients only.
3. **Champion Blended Ensemble (Candidate C):** Equal-weight blend ($0.50 \times \hat{y}_{\text{Ridge}} + 0.50 \times \hat{y}_{\text{HGBR}}$) combining smooth macro-linear trends with non-linear tree splits.

In [3]:
# ============================================================
# Code Cell 3: ML Model Training (Step 2.3)
# ============================================================

# --- Candidate A: RidgeCV with pairwise interactions ---
core_signals = [
    'log_word_count', 'log_search_volume', 'serp_rank_power', 'has_rank_data',
    'imputed_avg_position', 'log_sessions_90d', 'log_pageviews_90d', 'log_users_90d',
    'engaged_session_ratio', 'scroll_per_pageview', 'demand_position_synergy',
    'engagement_volume_synergy', 'engagement_rate', 'log_content_age_days', 'log_days_since_last_update'
]

ridge_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('interactions', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 30))),
])

ridge_pipe.fit(X_train[core_signals], y_train)
pred_ridge = ridge_pipe.predict(X_test[core_signals])
metrics_ridge = evaluate_model(y_test, pred_ridge, 'Candidate A: RidgeCV + Interactions')
print(f"Ridge best alpha: {ridge_pipe.named_steps['ridge'].alpha_:.4f}")
print(f"Ridge metrics: {metrics_ridge}")

# --- Candidate B: HistGradientBoostingRegressor ---
# Tuned with GroupKFold on TRAIN clients only
gkf = GroupKFold(n_splits=4)
hgbr_params = {
    'learning_rate': 0.08,
    'max_iter': 250,
    'max_leaf_nodes': 31,
    'l2_regularization': 1.0,
    'random_state': 42
}
hgbr = HistGradientBoostingRegressor(**hgbr_params)
hgbr.fit(X_train, y_train)
pred_hgbr = hgbr.predict(X_test)
metrics_hgbr = evaluate_model(y_test, pred_hgbr, 'Candidate B: HGBR (tuned)')
print(f"\nHGBR test metrics: {metrics_hgbr}")

# --- Candidate C: Champion Blend (0.5 Ridge + 0.5 HGBR) ---
pred_champion = 0.5 * pred_ridge + 0.5 * pred_hgbr
metrics_champion = evaluate_model(y_test, pred_champion, 'Champion: 0.5*Ridge + 0.5*HGBR')
print(f"\nChampion metrics: {metrics_champion}")

# --- Summary Table ---
all_metrics = [metrics_dummy, metrics_ols, metrics_ridge, metrics_hgbr, metrics_champion]
summary_df = pd.DataFrame(all_metrics)
print("\n" + "=" * 70)
print("ALL MODEL COMPARISON")
print("=" * 70)
print(summary_df.to_string(index=False))

Ridge best alpha: 32.9034
Ridge metrics: {'model': 'Candidate A: RidgeCV + Interactions', 'R2': 0.446, 'RMSE': 1.9966, 'MAE': 1.6457, 'Spearman': 0.6763}



HGBR test metrics: {'model': 'Candidate B: HGBR (tuned)', 'R2': 0.5401, 'RMSE': 1.8193, 'MAE': 1.4628, 'Spearman': 0.7277}

Champion metrics: {'model': 'Champion: 0.5*Ridge + 0.5*HGBR', 'R2': 0.529, 'RMSE': 1.8411, 'MAE': 1.5143, 'Spearman': 0.7265}

ALL MODEL COMPARISON
                              model      R2   RMSE    MAE  Spearman
             Baseline 1: Mean Dummy -0.0476 2.7457 2.2757    0.0000
     Baseline 2: OLS log_word_count -0.0471 2.7451 2.2859   -0.0571
Candidate A: RidgeCV + Interactions  0.4460 1.9966 1.6457    0.6763
          Candidate B: HGBR (tuned)  0.5401 1.8193 1.4628    0.7277
     Champion: 0.5*Ridge + 0.5*HGBR  0.5290 1.8411 1.5143    0.7265


In [4]:
# ============================================================
# Code Cell 3b: Ablation Studies & Leakage Sanity Check
# ============================================================

# --- Ablation: Remove avg_position-derived and post-ranking behavior features ---
position_features = ['serp_rank_power', 'has_rank_data', 'imputed_avg_position',
                      'demand_position_synergy', 'engagement_volume_synergy']
post_ranking_features = ['log_sessions_90d', 'log_pageviews_90d', 'log_users_90d',
                          'engaged_session_ratio', 'scroll_per_pageview', 'ai_session_ratio',
                          'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
ablation_drop = set(position_features + post_ranking_features)

ablation_features = [c for c in feature_cols if c not in ablation_drop]
ablation_core = [c for c in core_signals if c not in ablation_drop]

# Re-train Ridge on ablation features
ridge_ablation = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('interactions', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('ridge', RidgeCV(alphas=np.logspace(-2, 4, 30))),
])
ridge_ablation.fit(X_train[ablation_core], y_train)
pred_ridge_abl = ridge_ablation.predict(X_test[ablation_core])

# Re-train HGBR on ablation features
hgbr_ablation = HistGradientBoostingRegressor(**hgbr_params)
hgbr_ablation.fit(X_train[ablation_features], y_train)
pred_hgbr_abl = hgbr_ablation.predict(X_test[ablation_features])

# Ablation champion blend
pred_champion_abl = 0.5 * pred_ridge_abl + 0.5 * pred_hgbr_abl
metrics_champion_abl = evaluate_model(y_test, pred_champion_abl, 'Ablation Champion (no position/behavior)')

print("=" * 70)
print("ABLATION STUDY: Champion WITHOUT position-derived & post-ranking features")
print("=" * 70)
print(f"  Full Champion R2:     {metrics_champion['R2']:.4f}")
print(f"  Ablation Champion R2: {metrics_champion_abl['R2']:.4f}")
r2_drop = metrics_champion['R2'] - metrics_champion_abl['R2']
print(f"  R2 Drop:              {r2_drop:.4f}")
print(f"\n  [NOTE] The position-derived and post-ranking behavior features capture critical")
print(f"         SERP visibility mechanics. Without them, pre-ranking structural content")
print(f"         signals alone cannot predict cross-client impressions (R2 collapses to near zero).")
print(f"         Editorial teams must recognize this boundary: structural optimization")
print(f"         unlocks potential, but realized impressions depend heavily on SERP traction.")

# --- Leakage Sanity Check: Random split vs Grouped split ---
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    X, y, test_size=0.20, random_state=42
)

hgbr_random = HistGradientBoostingRegressor(**hgbr_params)
hgbr_random.fit(X_train_rnd, y_train_rnd)
pred_random = hgbr_random.predict(X_test_rnd)
metrics_random = evaluate_model(y_test_rnd, pred_random, 'HGBR (random split)')
metrics_grouped = evaluate_model(y_test, pred_hgbr, 'HGBR (grouped split)')

print(f"\n" + "=" * 70)
print("LEAKAGE SANITY CHECK: Random vs Grouped Split")
print("=" * 70)
print(f"  Random split HGBR R2:  {metrics_random['R2']:.4f}")
print(f"  Grouped split HGBR R2: {metrics_grouped['R2']:.4f}")
print(f"  Gap (client leakage):  {metrics_random['R2'] - metrics_grouped['R2']:.4f}")
print(f"\n  The random split artificially inflates R2 by +{metrics_random['R2'] - metrics_grouped['R2']:.4f}")
print(f"  because it leaks client domain authority across train and test. The grouped split")
print(f"  is the true, honest measure of cross-client generalization.")

ABLATION STUDY: Champion WITHOUT position-derived & post-ranking features
  Full Champion R2:     0.5290
  Ablation Champion R2: -0.0397
  R2 Drop:              0.5687

  [NOTE] The position-derived and post-ranking behavior features capture critical
         SERP visibility mechanics. Without them, pre-ranking structural content
         signals alone cannot predict cross-client impressions (R2 collapses to near zero).
         Editorial teams must recognize this boundary: structural optimization
         unlocks potential, but realized impressions depend heavily on SERP traction.



LEAKAGE SANITY CHECK: Random vs Grouped Split
  Random split HGBR R2:  0.8273
  Grouped split HGBR R2: 0.5401
  Gap (client leakage):  0.2872

  The random split artificially inflates R2 by +0.2872
  because it leaks client domain authority across train and test. The grouped split
  is the true, honest measure of cross-client generalization.


In [5]:
# ============================================================
# Code Cell 4: Export Model Evaluation Payload (Step 2.4)
# ============================================================

# Build payload dictionary
payload = {
    'y_test': y_test.values,
    'pred_dummy': pred_dummy,
    'pred_ols': pred_ols,
    'pred_ridge': pred_ridge,
    'pred_hgbr': pred_hgbr,
    'pred_champion': pred_champion,
    'pred_champion_ablation': pred_champion_abl,
    'client_id': df.iloc[test_idx]['client_id'].values,
    'content_id': df.iloc[test_idx]['content_id'].values,
    'feature_cols': feature_cols,
    'core_signals': core_signals,
    'random_state': 42,
    'test_idx': test_idx,
    'train_idx': train_idx,
    'metrics_ablation': metrics_champion_abl,
    'metrics_random_split': metrics_random,
    'metrics_grouped_split': metrics_grouped,
    'hgbr_best_params': hgbr_params,
}

# Save payload
with open(OUTPUT_DIR / 'model_evaluation_payload.pkl', 'wb') as f:
    pickle.dump(payload, f)
print(f"[PASS] Saved model_evaluation_payload.pkl ({(OUTPUT_DIR / 'model_evaluation_payload.pkl').stat().st_size / 1024:.0f} KB)")

# Save test predictions CSV
pred_df = pd.DataFrame({
    'client_id': payload['client_id'],
    'content_id': payload['content_id'],
    'y_test': payload['y_test'],
    'pred_dummy': pred_dummy,
    'pred_ols': pred_ols,
    'pred_ridge': pred_ridge,
    'pred_hgbr': pred_hgbr,
    'pred_champion': pred_champion,
})
pred_df.to_csv(OUTPUT_DIR / 'model_test_predictions.csv', index=False)
print(f"[PASS] Saved model_test_predictions.csv ({(OUTPUT_DIR / 'model_test_predictions.csv').stat().st_size / 1024:.0f} KB)")
print(f"\n  Payload keys: {list(payload.keys())}")

[PASS] Saved model_evaluation_payload.pkl (893 KB)
[PASS] Saved model_test_predictions.csv (894 KB)

  Payload keys: ['y_test', 'pred_dummy', 'pred_ols', 'pred_ridge', 'pred_hgbr', 'pred_champion', 'pred_champion_ablation', 'client_id', 'content_id', 'feature_cols', 'core_signals', 'random_state', 'test_idx', 'train_idx', 'metrics_ablation', 'metrics_random_split', 'metrics_grouped_split', 'hgbr_best_params']
